# OpenForensics Wild — generate one mask per image

This notebook does **not** train. It runs the current Full checkpoint on every
OpenForensics Wild image and saves the predicted mask.

The mask is the model’s sigmoid output, threshold 0.5, on the same crop the
trainer will use: largest Haar face, 10% margin, otherwise the whole frame,
letterboxed to 600×600. The crop box is stored in `index.csv` so training does
not run Haar again.

## Attach

1. New GPU notebook. Accelerator **GPU T4**. Internet Off is fine.
2. Dataset `ahmedgamal/openforensics-wild-dataset`.
3. The current Full checkpoint: `full_veritas_defacto_best.pt` or `.zip` (the `OF_04_full` output).
4. On every run after the first: **Add Input → Notebooks** → this notebook’s latest successful version.

## Where the files go

`/kaggle/working/ow_masks/`

| File | What it is |
| --- | --- |
| `manifest.csv` | Every Wild image, in a stable order |
| `index.csv` | Images this shard has finished. One row per image |
| `progress.json` | Count, images per hour, whether to resume |
| `masks/{train,val,test}/{real,fake}/{image_id}.png` | 600×600 mask, 0 or 255 |

`image_id` is the first 16 hex characters of SHA-1(`train/fake/filename`).
The same file name in two splits does not collide.

## Staying under 12 hours

`time_budget_hours` is **10.5**. The run stops before that, flushes `index.csv`,
and leaves the rest for the next session. Attach this output and Run All again
with the **same** `shard_index` and `shard_count`. Finished images are skipped.

After the first 64 new images the log prints images/hour and how many more runs
the remainder needs. One T4 pass is enough when that number is 1.

To use more than one GPU, set `shard_count` **before the first full run** and
start one session per shard (`shard_index` 0, 1, …). Do not change `shard_count`
after an `index.csv` exists. Training attaches every shard output.

Set `run_mode` to `full` for the real pass. `smoke` writes 8 masks only.


## 1. Setup


In [ ]:
import hashlib
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7

NOTEBOOK_START_TIME = time.time()
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


## 2. Find the Wild folders and the mask output


In [ ]:
def discover_wild_root() -> Path:
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if "ow_masks" in p.parts:
                continue
            kids = {d.lower() for d in dirnames}
            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:
                return p
            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):
                return p
    raise FileNotFoundError(
        "Attach ahmedgamal/openforensics-wild-dataset (Dataset/Train|Validation|Test/Real|Fake)."
    )


def _split_dir(root: Path, name: str) -> Path:
    aliases = {
        "train": ["Train", "train"],
        "val": ["Validation", "Val", "validation", "val"],
        "test": ["Test", "test"],
    }
    for a in aliases[name]:
        p = root / a
        if p.is_dir():
            return p
    raise FileNotFoundError(f"No {name} split under {root}")


WILD_ROOT = discover_wild_root()
IN_KAGGLE = Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "kaggle_working"
MASK_ROOT = WORKING / "ow_masks"
MASK_ROOT.mkdir(parents=True, exist_ok=True)
print("wild root:", WILD_ROOT)
print("mask root:", MASK_ROOT)


## 3. Configuration


In [ ]:
CONFIG = {
    "run_mode": "smoke",
    "image_size": 600,
    "batch_size": 4,
    "mask_threshold": 0.5,
    "face_margin": 0.10,
    "time_budget_hours": 10.5,
    "shard_index": 0,
    "shard_count": 1,
    "max_images": None,
    "flush_every": 20,
    "use_pretrained": False,
}
EXP_CFG = {"multi_stream": True, "segmentation": True, "legacy_baseline": False}
if CONFIG["run_mode"] == "smoke":
    CONFIG["max_images"] = 8
    CONFIG["batch_size"] = 2
    print("SMOKE: 8 images. Set run_mode='full' to walk every Wild image.")
if CONFIG["run_mode"] == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full mask generation needs GPU T4.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
print("device:", DEVICE)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 4. Manifest, finished rows, and the 10.5 hour ledger


In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LABELS = {"real": 0, "fake": 1}
INDEX_COLS = [
    "image_id", "rel_key", "split", "label", "class_name", "file_name",
    "crop_x0", "crop_y0", "crop_x1", "crop_y1", "face_found",
    "mask_file", "positive_frac", "status", "error", "teacher",
]


def rel_key(split: str, class_name: str, file_name: str) -> str:
    return f"{split}/{class_name}/{file_name}"


def image_id_for(key: str) -> str:
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]


def _class_dir(split_dir: Path, name: str):
    for a in (name, name.capitalize(), name.upper(), name.lower()):
        p = split_dir / a
        if p.is_dir():
            return p
    return None


def build_manifest() -> pd.DataFrame:
    rows = []
    for split in ("train", "val", "test"):
        split_dir = _split_dir(WILD_ROOT, split)
        for cname, lab in LABELS.items():
            cdir = _class_dir(split_dir, cname)
            if cdir is None:
                print("WARNING: missing", split, cname)
                continue
            for p in sorted(cdir.iterdir()):
                if p.is_file() and p.suffix.lower() in EXTS:
                    key = rel_key(split, cname, p.name)
                    rows.append({
                        "image_id": image_id_for(key),
                        "rel_key": key,
                        "split": split,
                        "label": lab,
                        "class_name": cname,
                        "file_name": p.name,
                        "src_path": str(p),
                    })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No Wild images found.")
    if df["image_id"].duplicated().any():
        raise RuntimeError("image_id collision; refuse to continue.")
    df = df.sort_values("rel_key").reset_index(drop=True)
    df.to_csv(MASK_ROOT / "manifest.csv", index=False)
    print("manifest", len(df), {s: int((df.split == s).sum()) for s in ("train", "val", "test")})
    return df


def _search_roots():
    roots = []
    for p in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), WORKING):
        if p.exists():
            roots.append(p)
    return roots


def load_prior() -> pd.DataFrame:
    frames = []
    seen = set()
    for root in _search_roots():
        for dirpath, _, filenames in os.walk(root):
            if "index.csv" not in filenames:
                continue
            path = Path(dirpath) / "index.csv"
            key = str(path.resolve()) if path.exists() else str(path)
            if key in seen:
                continue
            seen.add(key)
            try:
                df = pd.read_csv(path, dtype=str)
            except Exception as e:
                print("skip index", path, e)
                continue
            if not {"image_id", "rel_key", "mask_file", "status"} <= set(df.columns):
                continue
            df["_root"] = str(path.parent)
            frames.append(df)
            print("prior index", path, "rows", len(df))
    if not frames:
        return pd.DataFrame(columns=INDEX_COLS + ["_root"])
    return pd.concat(frames, ignore_index=True)


def png_for(row):
    p = Path(str(row["_root"])) / str(row["mask_file"])
    return p if p.is_file() else None


MANIFEST = build_manifest()
PRIOR = load_prior()
TEACHER_NAME = "pending"


## 5. Model (same Full VERITAS trunk as the current checkpoint)


In [ ]:
def _srm_kernels() -> torch.Tensor:
    k1 = np.array(
        [[0, 0, 0, 0, 0], [0, -1, 2, -1, 0], [0, 2, -4, 2, 0], [0, -1, 2, -1, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    k2 = np.array(
        [[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]],
        dtype=np.float32,
    )
    k3 = np.array(
        [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, -2, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    ks = []
    for k in (k1, k2, k3):
        s = np.abs(k).sum()
        ks.append(k / (s if s > 0 else 1.0))
    return torch.from_numpy(np.stack(ks, 0)[:, None, :, :])


class MultiStreamExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("srm_weight", _srm_kernels())

    def _denorm(self, x: torch.Tensor) -> torch.Tensor:
        mean = IMAGENET_MEAN.to(x.device, x.dtype)
        std = IMAGENET_STD.to(x.device, x.dtype)
        return (x * std + mean).clamp(0, 1)

    def forward(self, x_rgb_norm: torch.Tensor) -> torch.Tensor:
        rgb = self._denorm(x_rgb_norm)
        y = 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]
        residual = F.conv2d(y, self.srm_weight.to(dtype=rgb.dtype), padding=2).tanh()
        spec = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
        mag = torch.log1p(spec.abs())
        b, c, h, w = mag.shape
        mag_flat = mag.reshape(b, c, -1)
        mn = mag_flat.min(dim=-1, keepdim=True).values
        mx = mag_flat.max(dim=-1, keepdim=True).values
        mag = ((mag_flat - mn) / (mx - mn + 1e-6)).reshape(b, c, h, w)
        return torch.cat([rgb, residual, mag], dim=1)


class FeatureFusion(nn.Module):
    def __init__(self, conv3: nn.Conv2d):
        super().__init__()
        conv9 = nn.Conv2d(
            9, conv3.out_channels, kernel_size=conv3.kernel_size,
            stride=conv3.stride, padding=conv3.padding, bias=False,
        )
        with torch.no_grad():
            w = conv3.weight
            conv9.weight[:, 0:3].copy_(w)
            conv9.weight[:, 3:6].copy_(w)
            conv9.weight[:, 6:9].copy_(w)
            conv9.weight.mul_(1.0 / 3.0)
        self.conv = conv9

    def forward(self, x):
        return self.conv(x)


class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        self.branches.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        for r in rates:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        gn = 32 if out_ch % 32 == 0 else 1
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(gn, out_ch), nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True), nn.Dropout(0.1),
        )

    def forward(self, x):
        res = [b(x) for b in self.branches]
        gap = F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        res.append(gap)
        return self.project(torch.cat(res, dim=1))


class DeepLabV3PlusHead(nn.Module):
    def __init__(self, high_ch: int, low_ch: int, num_classes: int = 1):
        super().__init__()
        self.aspp = ASPP(high_ch, 256)
        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, high, low, out_size):
        high = F.interpolate(self.aspp(high), size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([high, self.low_proj(low)], dim=1)
        return F.interpolate(self.decoder(x), size=out_size, mode="bilinear", align_corners=False)


class ClassificationHead(nn.Module):
    def __init__(self, in_ch: int, dropout: float = 0.5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))

    def forward(self, feat):
        return self.fc(self.pool(feat).flatten(1))


class BaselineEffNet(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        net = efficientnet_b7(weights=weights)
        self.features = net.features
        self.cls_head = ClassificationHead(2560)
        self.multi_stream = False
        self.segmentation = False

    def forward(self, x):
        feat = self.features(x)
        logits = self.cls_head(feat)
        return {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}


class EfficientNetBackbone(nn.Module):
    def __init__(self, pretrained: bool = True, multi_stream: bool = False):
        super().__init__()
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        net = efficientnet_b7(weights=weights)
        stem = net.features[0]
        self.stem_rest = nn.Sequential(*list(stem.children())[1:])
        conv0 = stem[0]
        self.multi_stream = multi_stream
        self.stream = MultiStreamExtractor() if multi_stream else None
        if multi_stream:
            self.fusion = FeatureFusion(conv0)
            self.stem_conv = self.fusion.conv
        else:
            self.fusion = None
            self.stem_conv = conv0
        self.stages = nn.ModuleList(list(net.features.children())[1:])
        self.out_channels = 2560
        self.low_stage = 1
        self.high_stage = 4

    def forward(self, x):
        x = self.stem_conv(self.stream(x) if self.multi_stream else x)
        x = self.stem_rest(x)
        low = high = None
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i == self.low_stage:
                low = x
            if i == self.high_stage:
                high = x
        return {"final": x, "low": low if low is not None else x, "high": high if high is not None else x}


class VERITASModel(nn.Module):
    def __init__(self, multi_stream=False, segmentation=False, pretrained=True):
        super().__init__()
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.encoder = EfficientNetBackbone(pretrained=pretrained, multi_stream=self.multi_stream)
        self.cls_head = ClassificationHead(self.encoder.out_channels)
        self.seg_head = DeepLabV3PlusHead(224, 48, 1) if self.segmentation else None

    def forward(self, x):
        feats = self.encoder(x)
        logits = self.cls_head(feats["final"])
        out = {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}
        if self.segmentation:
            seg_logits = self.seg_head(feats["high"], feats["low"], out_size=x.shape[-2:])
            out["segmentation_logits"] = seg_logits
            out["segmentation_prob"] = torch.sigmoid(seg_logits)
        return out


def build_model():
    if EXP_CFG.get("legacy_baseline"):
        return BaselineEffNet(pretrained=CONFIG["use_pretrained"]).to(DEVICE)
    m = VERITASModel(
        multi_stream=EXP_CFG["multi_stream"],
        segmentation=EXP_CFG["segmentation"],
        pretrained=CONFIG["use_pretrained"],
    ).to(DEVICE)
    if m.segmentation:
        was_training = m.training
        m.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 64, 64, device=DEVICE)
            feats = m.encoder(dummy)
            low_ch, high_ch = feats["low"].shape[1], feats["high"].shape[1]
        if m.seg_head.low_proj[0].in_channels != low_ch or m.seg_head.aspp.branches[0][0].in_channels != high_ch:
            m.seg_head = DeepLabV3PlusHead(high_ch, low_ch, 1).to(DEVICE)
            print(f"Adapted DeepLabV3+ head to low_ch={low_ch} high_ch={high_ch}")
        if was_training:
            m.train()
    return m


## 6. Load the current Full checkpoint


In [ ]:
def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def materialize_checkpoint(path: Path) -> Path:
    path = Path(path)
    try:
        _torch_load(path)
        return path
    except Exception:
        pass
    if path.suffix.lower() != ".zip":
        raise RuntimeError(f"Cannot read checkpoint: {path}")
    cache = WORKING / (path.stem.replace(" ", "_") + "_repacked.pt")
    if cache.exists() and cache.stat().st_size > 1_000_000:
        return cache
    with zipfile.ZipFile(path) as zin:
        names = zin.namelist()
        pkl = next((n for n in names if n.endswith("data.pkl")), None)
        if pkl is None:
            raise RuntimeError("Zip is not a PyTorch checkpoint (no data.pkl)")
        prefix = pkl[: -len("data.pkl")]
        with zipfile.ZipFile(cache, "w", compression=zipfile.ZIP_STORED) as zout:
            for name in names:
                if prefix and not name.startswith(prefix):
                    continue
                inner = name[len(prefix):]
                if not inner or inner.endswith("/"):
                    continue
                zout.writestr(inner, zin.read(name))
    return cache


def find_teacher() -> Path:
    order = [
        "full_veritas_defacto_best.pt",
        "full_veritas_defacto_best.zip",
        "full_veritas_defacto_last.pt",
    ]
    found = {name: [] for name in order}
    for root in _search_roots():
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                if fn in found:
                    found[fn].append(Path(dirpath) / fn)
    for name in order:
        if found[name]:
            return found[name][0]
    raise FileNotFoundError(
        "Attach full_veritas_defacto_best.pt or .zip from OF_04_full. "
        "The baseline checkpoint has no mask head."
    )


teacher_path = find_teacher()
print("teacher file:", teacher_path)
ck = _torch_load(materialize_checkpoint(teacher_path))
state = ck["model_state_dict"] if isinstance(ck, dict) and "model_state_dict" in ck else ck
exp_cfg = (ck.get("exp_cfg") if isinstance(ck, dict) else None) or {}
if exp_cfg and (not exp_cfg.get("segmentation", True) or not exp_cfg.get("multi_stream", True)):
    raise RuntimeError(f"Checkpoint is not Full VERITAS: {exp_cfg}")
model = build_model()
model.load_state_dict(state, strict=True)
model.eval()
TEACHER_NAME = teacher_path.name
print("loaded", TEACHER_NAME, "epoch", None if not isinstance(ck, dict) else ck.get("epoch"))


## 7. Paint masks until the time budget


In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_HAAR = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def letterbox_rgb(image_rgb, size: int):
    h, w = image_rgb.shape[:2]
    scale = min(size / max(h, 1), size / max(w, 1))
    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)
    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas[top: top + nh, left: left + nw] = resized
    return canvas


def bbox_from_mask(mask, margin: float):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    h, w = mask.shape[:2]
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    bw, bh = max(x1 - x0, 1), max(y1 - y0, 1)
    pad_x, pad_y = int(round(bw * margin)), int(round(bh * margin))
    x0, y0 = max(0, x0 - pad_x), max(0, y0 - pad_y)
    x1, y1 = min(w, x1 + pad_x), min(h, y1 + pad_y)
    return x0, y0, x1, y1


def bbox_from_haar(rgb, margin: float):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = _HAAR.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(48, 48))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    dummy = np.zeros(rgb.shape[:2], dtype=np.uint8)
    dummy[y: y + h, x: x + w] = 1
    return bbox_from_mask(dummy, margin)


def shard_of(image_id: str) -> int:
    return int(image_id[:8], 16) % int(CONFIG["shard_count"])


def prior_done_for_shard() -> dict:
    done = {}
    if PRIOR.empty:
        return done
    teachers = set()
    for rec in PRIOR.to_dict("records"):
        if str(rec.get("status")) != "ok":
            continue
        iid = str(rec["image_id"])
        if shard_of(iid) != int(CONFIG["shard_index"]):
            continue
        png = png_for(rec)
        if png is None:
            continue
        teacher = str(rec.get("teacher") or "")
        if teacher and teacher not in ("nan", "None"):
            teachers.add(teacher)
        done[iid] = rec
    if len(teachers) > 1:
        raise RuntimeError(f"Prior masks come from more than one checkpoint: {teachers}")
    if teachers and TEACHER_NAME not in teachers and "pending" not in teachers:
        # The file name of a resumed zip and a .pt can differ. Compare the stem family.
        def family(name: str) -> str:
            return name.replace("_repacked", "").replace(".zip", "").replace(".pt", "")
        if {family(t) for t in teachers} != {family(TEACHER_NAME)}:
            raise RuntimeError(
                f"Prior masks were painted by {teachers}, this run loaded {TEACHER_NAME}. "
                "Do not mix checkpoints in one mask set."
            )
    return done


DONE = prior_done_for_shard()
SEEN = set(DONE)
if not PRIOR.empty:
    for rec in PRIOR.to_dict("records"):
        iid = str(rec.get("image_id") or "")
        if not iid or shard_of(iid) != int(CONFIG["shard_index"]):
            continue
        if str(rec.get("status")) == "error":
            SEEN.add(iid)
print("already finished in this shard:", len(DONE), "| rows that will not be retried:", len(SEEN))

copied = 0
for iid, rec in DONE.items():
    src = png_for(rec)
    dst = MASK_ROOT / str(rec["mask_file"])
    if src is None or dst.exists() or src.resolve() == dst.resolve():
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied += 1
print("copied earlier masks into this output:", copied)


def write_index(rows: list):
    df = pd.DataFrame(rows)
    for col in INDEX_COLS:
        if col not in df.columns:
            df[col] = ""
    df = df[INDEX_COLS].drop_duplicates("image_id", keep="last")
    tmp = MASK_ROOT / "index.csv.tmp"
    df.to_csv(tmp, index=False)
    tmp.replace(MASK_ROOT / "index.csv")
    return df


def write_progress(payload: dict):
    tmp = MASK_ROOT / "progress.json.tmp"
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    tmp.replace(MASK_ROOT / "progress.json")


shard_df = MANIFEST[MANIFEST.image_id.map(shard_of) == int(CONFIG["shard_index"])].reset_index(drop=True)
if CONFIG.get("max_images"):
    shard_df = shard_df.iloc[: int(CONFIG["max_images"])].reset_index(drop=True)
todo = shard_df[~shard_df.image_id.isin(SEEN)].reset_index(drop=True)
print(f"this shard {CONFIG['shard_index']}/{CONFIG['shard_count']}: {len(shard_df)} images, {len(todo)} still to paint")

budget_s = float(CONFIG["time_budget_hours"]) * 3600.0
size = int(CONFIG["image_size"])
thresh = float(CONFIG["mask_threshold"])
margin = float(CONFIG["face_margin"])
rows = []
for rec in DONE.values():
    item = {k: rec.get(k, "") for k in INDEX_COLS}
    item["teacher"] = TEACHER_NAME
    rows.append(item)
if not PRIOR.empty:
    for rec in PRIOR.to_dict("records"):
        iid = str(rec.get("image_id") or "")
        if iid in DONE or shard_of(iid) != int(CONFIG["shard_index"]):
            continue
        if str(rec.get("status")) == "error":
            item = {k: rec.get(k, "") for k in INDEX_COLS}
            item["teacher"] = TEACHER_NAME
            rows.append(item)

pending = []
n_new = 0
stopped = "finished"
t_paint = time.time()


def flush_batch():
    global n_new
    if not pending:
        return
    batch = torch.stack([p["tensor"] for p in pending]).to(DEVICE)
    with torch.no_grad():
        if DEVICE.type == "cuda":
            with torch.amp.autocast("cuda"):
                out = model(batch)
        else:
            out = model(batch)
    if "segmentation_prob" not in out:
        raise RuntimeError("Loaded model has no segmentation head.")
    probs = out["segmentation_prob"].detach().float().cpu().numpy()
    for i, item in enumerate(pending):
        # Index rows live for the whole run; never retain image tensors in them.
        del item["tensor"]
        pred = (probs[i, 0] >= thresh).astype(np.uint8) * 255
        dst = MASK_ROOT / item["mask_file"]
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not cv2.imwrite(str(dst), pred):
            rows.append({**item, "positive_frac": "", "status": "error", "error": "imwrite failed", "teacher": TEACHER_NAME})
        else:
            rows.append({
                **item,
                "positive_frac": f"{float((pred > 0).mean()):.6f}",
                "status": "ok",
                "error": "",
                "teacher": TEACHER_NAME,
            })
        n_new += 1
    pending.clear()
    del batch, out, probs


def record_error(row, message):
    rows.append({
        "image_id": row.image_id,
        "rel_key": row.rel_key,
        "split": row.split,
        "label": int(row.label),
        "class_name": row.class_name,
        "file_name": row.file_name,
        "crop_x0": "", "crop_y0": "", "crop_x1": "", "crop_y1": "",
        "face_found": 0,
        "mask_file": "",
        "positive_frac": "",
        "status": "error",
        "error": message,
        "teacher": TEACHER_NAME,
    })


for n_seen, row in enumerate(todo.itertuples(index=False), start=1):
    if time.time() - NOTEBOOK_START_TIME > budget_s:
        stopped = "time_budget"
        break
    bgr = cv2.imread(row.src_path, cv2.IMREAD_COLOR)
    if bgr is None:
        record_error(row, "unreadable")
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    box = bbox_from_haar(rgb, margin)
    face_found = 1 if box is not None else 0
    if box is None:
        box = (0, 0, rgb.shape[1], rgb.shape[0])
    x0, y0, x1, y1 = box
    crop = rgb[y0:y1, x0:x1]
    if crop.size == 0:
        record_error(row, "empty crop")
        continue
    canvas = letterbox_rgb(crop, size)
    x = canvas.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN_NP) / IMAGENET_STD_NP
    tensor = torch.from_numpy(np.transpose(x, (2, 0, 1)).copy())
    pending.append({
        "tensor": tensor,
        "image_id": row.image_id,
        "rel_key": row.rel_key,
        "split": row.split,
        "label": int(row.label),
        "class_name": row.class_name,
        "file_name": row.file_name,
        "crop_x0": int(x0), "crop_y0": int(y0), "crop_x1": int(x1), "crop_y1": int(y1),
        "face_found": face_found,
        "mask_file": f"masks/{row.split}/{row.class_name}/{row.image_id}.png",
    })
    if len(pending) >= int(CONFIG["batch_size"]):
        flush_batch()
    if n_new and n_new % int(CONFIG["flush_every"]) == 0 and not pending:
        write_index(rows)
        rate = n_new / max(time.time() - t_paint, 1e-6)
        remain = len(todo) - n_seen
        print(
            f"painted {n_new} this run | shard done {len(DONE) + n_new}/{len(shard_df)} | "
            f"{rate * 3600:.0f} images/hour | remaining hours {remain / rate / 3600:.2f}"
        )
        if n_new == int(CONFIG["flush_every"]) or n_new == 64:
            hours_needed = (len(todo) / rate) / 3600.0
            runs = int(np.ceil(hours_needed / float(CONFIG["time_budget_hours"])))
            print(f"estimate for the images that were still queued at the start: {hours_needed:.2f} h, about {runs} run(s) at this rate")
            if runs > 1 and int(CONFIG["shard_count"]) == 1 and len(DONE) == 0:
                print(
                    "This shard will not finish in one session. Let this run hit the time budget, "
                    "then attach this output and Run All again with the same shard_index and shard_count. "
                    "Do not change shard_count after index.csv exists."
                )

flush_batch()
index_df = write_index(rows)
elapsed_h = (time.time() - NOTEBOOK_START_TIME) / 3600.0
n_ok = int((index_df.status == "ok").sum()) if len(index_df) else 0
rate = n_new / max(time.time() - t_paint, 1e-6)
remain = max(len(shard_df) - n_ok, 0)
payload = {
    "teacher": TEACHER_NAME,
    "shard_index": int(CONFIG["shard_index"]),
    "shard_count": int(CONFIG["shard_count"]),
    "n_manifest": int(len(MANIFEST)),
    "n_shard": int(len(shard_df)),
    "n_ok_this_shard": n_ok,
    "n_painted_this_run": int(n_new),
    "images_per_hour": None if n_new == 0 else round(rate * 3600.0, 1),
    "hours_used": round(elapsed_h, 3),
    "remaining_this_shard": int(remain),
    "stopped_reason": stopped if remain else "finished",
    "resume": remain > 0,
}
write_progress(payload)
print(json.dumps(payload, indent=2))
if CONFIG["run_mode"] == "smoke":
    print("SMOKE only painted a handful of images. progress.json does not mean the full Wild set is done.")
elif payload["resume"]:
    print("RESUME: attach this notebook output and Run All again. Keep shard_index and shard_count.")
else:
    print("SHARD COMPLETE. Training attaches this output (and the other shard outputs, if shard_count > 1).")
